# Valutazione del Framework BB84 AI Red Teaming

**Notebook interattivo per l'analisi dei risultati della valutazione multi-agente.**

Questo notebook permette di:
1. Esecuzione diretta delle trial (con o senza LLM)
2. Caricamento dei risultati da file JSON/CSV
3. Analisi statistica per RQ1, RQ3, RQ5
4. Generazione di grafici per la tesi
5. Esplorazione interattiva dei dati

## Dipendenze richieste
```bash
pip install pandas numpy matplotlib seaborn scipy
```

## Struttura del modulo
- `evaluation/trial_logger.py` – TrialLogger: cattura dati di una singola run
- `evaluation/evaluation_runner.py` – EvaluationRunner: orchestratore delle run
- `evaluation/analysis.py` – compute_metrics() e analisi per RQ
- `evaluation/plotting.py` – Funzioni di visualizzazione
- `evaluation/attack_scenarios.py` – Scenari d'attacco predefiniti

In [1]:
# Importazioni base
import sys
import os
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Aggiungi la root del progetto a sys.path
project_root = Path.cwd().resolve()
# Il notebook è in evaluation/, quindi aggiungi sia la cartella corrente che il genitore
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root.parent) not in sys.path:
    sys.path.insert(0, str(project_root.parent))

# Import del modulo evaluation
from evaluation.trial_logger import TrialResult
from evaluation.evaluation_runner import EvaluationRunner, EvaluationConfig, run_evaluation
from evaluation.analysis import (
    compute_metrics, compute_rq1, compute_rq3, compute_rq5,
    compute_scenario_comparison, compute_correlations,
    generate_report, save_report, load_results
)
from evaluation.plotting import (
    plot_success_rate, plot_qber_distribution, plot_qber_by_scenario_box,
    plot_stealth_vs_success, plot_agent_collaboration, plot_handoff_analysis,
    plot_confidence_analysis, plot_robustness_analysis, plot_time_distribution,
    plot_attack_comparison, plot_correlation_matrix, plot_scenario_summary,
    generate_all_plots,
)
from evaluation.attack_scenarios import SCENARIOS, get_all_scenarios

import numpy as np
import pandas as pd

print("[OK] Tutte le importazioni caricate con successo.")
print(f"[INFO] Project root: {project_root}")

[OK] Tutte le importazioni caricate con successo.
[INFO] Project root: D:\Unsloth\TESI_FINALE\evaluation


## 1. Configurazione

Modifica i parametri qui sotto per controllare la valutazione.

In [2]:
# Parametri di configurazione
NUM_TRIALS = 50           # Trial per scenario
SEED = 42                 # Seed per riproducibilità
SCENARIOS_TO_RUN = list(SCENARIOS.keys())  # ['intercept_resend', 'pns', 'blinding', ...]
OUTPUT_DIR = str(Path.cwd().resolve() / "output")
PLOTS_DIR = str(Path.cwd().resolve() / "plots")

print(f"Configurazione:")
print(f"  Trial per scenario: {NUM_TRIALS}")
print(f"  Seed: {SEED}")
print(f"  Scenari: {SCENARIOS_TO_RUN}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Grafici: {PLOTS_DIR}")
print()
print("Scenari disponibili:")
for name, scenario in get_all_scenarios().items():
    print(f"  [{name}] {scenario.description}")

Configurazione:
  Trial per scenario: 50
  Seed: 42
  Scenari: ['intercept_resend', 'intercept_resend_stealth', 'pns', 'blinding', 'trojan_horse', 'qber_tamper', 'mixed', 'clean_baseline']
  Output: D:\Unsloth\TESI_FINALE\evaluation\output
  Grafici: D:\Unsloth\TESI_FINALE\evaluation\plots

Scenari disponibili:
  [intercept_resend] Attacco Intercept-Resend base: Eve misura e rispedisce qubit
  [intercept_resend_stealth] Attacco Intercept-Resend ottimizzato: interception_rate basso per minimizzare QBER
  [pns] PNS attack (Photon Number Splitting): Eve blocca pulsazioni a singolo fotone
  [blinding] Blinding attack + QBER tampering: Eve controlla completamente il canale
  [trojan_horse] Trojan Horse attack: Eve inietta luce per rivelare la base di Alice
  [qber_tamper] QBER tampering: Eve altera la stima QBER per evitare il rilevamento
  [mixed] Attacco randomizzato: combina multiple strategie con probabilità per iterazione
  [clean_baseline] Baseline: nessun attacco Eve, solo rumore di 

## 2. Esecuzione della valutazione

Due opzioni:
- **Opzione A**: Esegui le trial (richiede che il simulatore BB84 sia importabile)
- **Opzione B**: Carica dati esistenti da file JSON

### Opzione A: Esegui le trial (Approccio Ibrido)

In [3]:
# ============================================================================
# APPROCCIO IBRIDO: Valutazione simulatore + opzionale flusso agenti
# ============================================================================
#
# Modalità 1 - Simulatore diretto (default, veloce):
#   Esegue BB84SimulationV2 direttamente per tutti gli scenari.
#   Adatto per RQ1 (Attack Effectiveness) e RQ5 (Robustness).
#
# Modalità 2 - Ibrida (opzionale, lenta):
#   Per un subset di trial, esegue il flusso completo:
#   Recon Agent → Planning Agent → Execution Agent → BB84SimulationV2
#   Adatto per RQ3 (Multi-Agent Collaboration Quality).
#   Richiede LM Studio attivo e dati di planning nel database.
# ============================================================================

# CONFIGURAZIONE APPROCCIO IBRIDO
USE_HYBRID_MODE = True  # True = esegue anche trial ibride, False = solo simulatore
HYBRID_AGENT_TRIALS = 5  # Numero di trial ibride per scenario (se USE_HYBRID_MODE=True)

print("=" * 70)
print("APPROCCIO IBRIDO - CONFIGURAZIONE")
print("=" * 70)
print(f"Modalità simulatore diretto: {'SÌ' if not USE_HYBRID_MODE else 'NO'}")
print(f"Modalità ibrida (agenti):     {'SÌ' if USE_HYBRID_MODE else 'NO'}")
if USE_HYBRID_MODE:
    print(f"Trial ibride per scenario:    {HYBRID_AGENT_TRIALS}")
    print(f"\n[ATTENZIONE] LM Studio deve essere attivo su http://localhost:1234")
print(f"Trial totali previste:        {NUM_TRIALS * len(SCENARIOS_TO_RUN)}")
if USE_HYBRID_MODE:
    print(f"Trial agenti extra:           {HYBRID_AGENT_TRIALS * len(SCENARIOS_TO_RUN)}")
print("=" * 70)

APPROCCIO IBRIDO - CONFIGURAZIONE
Modalità simulatore diretto: NO
Modalità ibrida (agenti):     SÌ
Trial ibride per scenario:    5

[ATTENZIONE] LM Studio deve essere attivo su http://localhost:1234
Trial totali previste:        400
Trial agenti extra:           40


In [4]:
# Esegui la valutazione
try:
    runner = EvaluationRunner(EvaluationConfig(
        num_trials=NUM_TRIALS,
        seed=SEED,
        base_seed=SEED,
        scenario_names=SCENARIOS_TO_RUN,
        output_dir=OUTPUT_DIR,
        plots_dir=PLOTS_DIR,
        hybrid_agent_trials=HYBRID_AGENT_TRIALS if USE_HYBRID_MODE else 0,
    ))
    
    results = runner.run_all()
    print(f"\n[OK] Valutazione completata: {len(results)} trial eseguite.")
    
    # Salva risultati
    runner.logger.save_results_json()
    runner.logger.save_metrics_csv()
    
    print(f"[OK] Risultati salvati in:")
    print(f"  JSON: {runner.logger.results_path}")
    print(f"  CSV:  {runner.logger.metrics_path}")
    
    # Summary
    summary = runner.logger.get_summary()
    print(f"\n[Summary]")
    print(f"  Success rate: {summary.get('success_rate', 0):.2%}")
    print(f"  Crash rate:   {summary.get('crash_rate', 0):.2%}")
    print(f"  Avg QBER:     {summary.get('avg_qber', 0):.4f}")
    print(f"  Avg tempo:    {summary.get('avg_wall_time', 0):.2f}s")
    
except Exception as e:
    print(f"[ERRORE] Durante l'esecuzione: {e}")
    import traceback
    traceback.print_exc()


[EvaluationRunner] Avvio valutazione: 8 scenari, 50 trial ciascuno, seed=42
[EvaluationRunner] Output: D:\Unsloth\TESI_FINALE\evaluation\output

Scenario: intercept_resend
  Attacco: Attacco Intercept-Resend base: Eve misura e rispedisce qubit
  [intercept_resend] trial 1/50: FAIL | QBER=0.1379 (true=0.1379) | Tempo=0.13s
  [intercept_resend] trial 2/50: FAIL | QBER=0.1395 (true=0.1395) | Tempo=0.15s
  [intercept_resend] trial 3/50: FAIL | QBER=0.1468 (true=0.1468) | Tempo=0.11s
  [intercept_resend] trial 4/50: FAIL | QBER=0.1612 (true=0.1612) | Tempo=0.12s
  [intercept_resend] trial 5/50: FAIL | QBER=0.1673 (true=0.1673) | Tempo=0.11s
  [intercept_resend] trial 6/50: FAIL | QBER=0.1130 (true=0.1130) | Tempo=0.11s
  [intercept_resend] trial 7/50: FAIL | QBER=0.1545 (true=0.1545) | Tempo=0.11s
  [intercept_resend] trial 8/50: FAIL | QBER=0.1602 (true=0.1602) | Tempo=0.15s
  [intercept_resend] trial 9/50: SUCCESS | QBER=0.1079 (true=0.1079) | Tempo=0.11s
  [intercept_resend] trial 10/50

### Opzione B: Carica dati esistenti

In [5]:
# Carica i risultati da file JSON
results_json_path = Path(OUTPUT_DIR) / "evaluation_results.json"

if results_json_path.exists():
    df = load_results(str(results_json_path))
    print(f"[OK] Caricati {len(df)} trial da {results_json_path}")
else:
    print(f"[INFO] File non trovato: {results_json_path}")
    print("[INFO] Esegui prima la valutazione (Opzione A) o carica un file CSV.")
    df = pd.DataFrame()

[OK] Caricati 400 trial da D:\Unsloth\TESI_FINALE\evaluation\output\evaluation_results.json


## 3. Esplorazione dei dati

In [6]:
if df.empty:
    print("Nessun dato da esplorare.")
else:
    # Info generali
    print("=" * 60)
    print("INFO GENERALI")
    print("=" * 60)
    print(f"Trial totali: {len(df)}")
    print(f"Scenario: {df['scenario_name'].nunique()}")
    print(f"Scenari: {df['scenario_name'].unique()}")
    print(f"Success rate globale: {(~df['crashed'] & df['success']).mean():.2%}")
    print(f"Tasso crash: {df['crashed'].mean():.2%}")
    print()
    
    # Prime righe
    print("PRIME RIGHE DEI DATI:")
    display_cols = ['run_id', 'seed', 'scenario_name', 'attack_type', 'success', 
                    'qber', 'stolen_key_bits', 'wall_time_sec', 'crashed']
    available_cols = [c for c in display_cols if c in df.columns]
    print(df[available_cols].head(10).to_string(index=False))
    print()
    
    # Statistiche descrittive
    print("STATISTICHE DESCRITTIVE:")
    numeric_cols = ['qber', 'stolen_key_bits', 'wall_time_sec', 'agent_messages',
                    'handoffs_attempted', 'handoffs_successful', 'secure_key_length']
    available_numeric = [c for c in numeric_cols if c in df.columns]
    print(df[available_numeric].describe().round(4))

INFO GENERALI
Trial totali: 400
Scenario: 8
Scenari: ['intercept_resend' 'intercept_resend_stealth' 'pns' 'blinding'
 'trojan_horse' 'qber_tamper' 'mixed' 'clean_baseline']
Success rate globale: 46.50%
Tasso crash: 0.00%

PRIME RIGHE DEI DATI:
         run_id  seed    scenario_name      attack_type  success     qber  stolen_key_bits  wall_time_sec  crashed
intercept__0000   550 intercept_resend intercept_resend    False 0.137908                0       0.125414    False
intercept__0001  1550 intercept_resend intercept_resend    False 0.139546                0       0.151187    False
intercept__0002  2550 intercept_resend intercept_resend    False 0.146844                0       0.111864    False
intercept__0003  3550 intercept_resend intercept_resend    False 0.161223                0       0.119675    False
intercept__0004  4550 intercept_resend intercept_resend    False 0.167320                0       0.114278    False
intercept__0005  5550 intercept_resend intercept_resend    False 0

## 4. Analisi per Domanda di Ricerca

### RQ1 – Attack Effectiveness

In [7]:
if not df.empty:
    rq1 = compute_rq1(df)
    print(json.dumps(rq1, indent=2, default=str))
else:
    print("Nessun dato per RQ1.")

{
  "blinding": {
    "total_trials": 50,
    "valid_trials": 50,
    "success_rate": 0.0,
    "stealth_rate": 1.0,
    "qber_mean": 0.13890032000000002,
    "qber_std": 0.03750069334294545,
    "qber_min": 0.065566,
    "qber_max": 0.206664,
    "qber_median": 0.14089849999999998,
    "avg_stolen_bits": 0.0,
    "key_compromised_rate": 0.0,
    "detected_rate": 0.0,
    "eve_info_mean": 0.0
  },
  "clean_baseline": {
    "total_trials": 50,
    "valid_trials": 50,
    "success_rate": 1.0,
    "stealth_rate": 0.9,
    "qber_mean": 0.07012348000000002,
    "qber_std": 0.02807309145510217,
    "qber_min": 0.02386,
    "qber_max": 0.144002,
    "qber_median": 0.0703435,
    "avg_stolen_bits": 75.14,
    "key_compromised_rate": 0.9,
    "detected_rate": 0.1,
    "eve_info_mean": 89.0
  },
  "intercept_resend": {
    "total_trials": 50,
    "valid_trials": 50,
    "success_rate": 0.32,
    "stealth_rate": 0.32,
    "qber_mean": 0.12996404,
    "qber_std": 0.033007354984918254,
    "qber_min

### RQ3 – Multi-Agent Collaboration Quality

In [8]:
if not df.empty:
    rq3 = compute_rq3(df)
    print(json.dumps(rq3, indent=2, default=str))
else:
    print("Nessun dato per RQ3.")

{
  "blinding": {
    "total_trials": 50,
    "valid_trials": 50,
    "avg_agent_messages": 0.0,
    "avg_handoffs_attempted": 0.0,
    "avg_handoffs_successful": 0.0,
    "handoff_rate": 0.0,
    "avg_wall_time": 0.12444011688232422,
    "time_std": 0.011073373710162909,
    "time_cv": 0.08898556179141454,
    "avg_llm_calls": 0.0
  },
  "clean_baseline": {
    "total_trials": 50,
    "valid_trials": 50,
    "avg_agent_messages": 0.0,
    "avg_handoffs_attempted": 0.0,
    "avg_handoffs_successful": 0.0,
    "handoff_rate": 0.0,
    "avg_wall_time": 0.08123749256134033,
    "time_std": 0.011525209741766044,
    "time_cv": 0.14187057451414636,
    "avg_llm_calls": 0.0
  },
  "intercept_resend": {
    "total_trials": 50,
    "valid_trials": 50,
    "avg_agent_messages": 0.0,
    "avg_handoffs_attempted": 0.0,
    "avg_handoffs_successful": 0.0,
    "handoff_rate": 0.0,
    "avg_wall_time": 0.11921729564666748,
    "time_std": 0.011777663085143903,
    "time_cv": 0.09879156393590889,
   

### RQ5 – Robustness and Reproducibility

In [9]:
if not df.empty:
    rq5 = compute_rq5(df)
    print(json.dumps(rq5, indent=2, default=str))
else:
    print("Nessun dato per RQ5.")

{
  "blinding": {
    "total_trials": 50,
    "valid_trials": 50,
    "crash_rate": 0.0,
    "success_cv": 0.0,
    "time_cv": 0.08898556179141454,
    "qber_cv": 0.2699827714071893,
    "avg_wall_time": 0.12444011688232422,
    "time_std": 0.011073373710162909,
    "success_mean": 0.0,
    "success_std": 0.0
  },
  "clean_baseline": {
    "total_trials": 50,
    "valid_trials": 50,
    "crash_rate": 0.0,
    "success_cv": 0.0,
    "time_cv": 0.14187057451414636,
    "qber_cv": 0.4003379674697001,
    "avg_wall_time": 0.08123749256134033,
    "time_std": 0.011525209741766044,
    "success_mean": 1.0,
    "success_std": 0.0
  },
  "intercept_resend": {
    "total_trials": 50,
    "valid_trials": 50,
    "crash_rate": 0.0,
    "success_cv": 1.4725377234348787,
    "time_cv": 0.09879156393590889,
    "qber_cv": 0.2539729834877267,
    "avg_wall_time": 0.11921729564666748,
    "time_std": 0.011777663085143903,
    "success_mean": 0.32,
    "success_std": 0.47121207149916117
  },
  "interce

## 5. Metriche complete

In [10]:
if not df.empty:
    metrics = compute_metrics(df)
    
    # Report testuale
    report = generate_report(metrics, df)
    print(report)
    
    # Salva report
    report_path = save_report(metrics, df, OUTPUT_DIR)
    print(f"\n[OK] Report salvato in: {report_path}")
else:
    print("Nessun dato per le metriche complete.")

RAPPORTO DI VALUTAZIONE BB84 AI RED TEAMING

SINTESI GENERALE
----------------------------------------
  Trial totali: 400
  Trial riuscite: 186
  Trial fallite: 214
  Crash: 0
  Success rate globale: 46.50%

RQ1 – ATTACK EFFECTIVENESS
----------------------------------------
  [clean_baseline]
    Success rate: 100.00%
    Stealth rate: 90.00%
    QBER mean ± std: 0.0701 ± 0.0281
    Avg stolen bits: 75.1
    Detected rate: 10.00%

  [pns]
    Success rate: 100.00%
    Stealth rate: 100.00%
    True QBER mean ± std: 0.0575 ± 0.0194
    Avg stolen bits: 77.6
    Detected rate: 0.00%

  [intercept_resend_stealth]
    Success rate: 82.00%
    Stealth rate: 82.00%
    True QBER mean ± std: 0.0850 ± 0.0272
    Avg stolen bits: 78.3
    Detected rate: 18.00%

  [trojan_horse]
    Success rate: 58.00%
    Stealth rate: 58.00%
    True QBER mean ± std: 0.1100 ± 0.0325
    Avg stolen bits: 61.2
    Detected rate: 42.00%

  [intercept_resend]
    Success rate: 32.00%
    Stealth rate: 32.00%
  

## 6. Visualizzazioni

### Grafici per RQ1 (Attack Effectiveness)

In [12]:
if not df.empty:
    print("Generazione grafici RQ1...")
    
    p1 = plot_success_rate(df, PLOTS_DIR)
    print(f"  [OK] {p1}")
    
    p2 = plot_qber_distribution(df, PLOTS_DIR)
    print(f"  [OK] {p2}")
    
    p3 = plot_qber_by_scenario_box(df, PLOTS_DIR)
    print(f"  [OK] {p3}")
    
    p4 = plot_stealth_vs_success(df, PLOTS_DIR)
    print(f"  [OK] {p4}")
else:
    print("Nessun dato per i grafici.")

Generazione grafici RQ1...
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\rq1_success_rate.pdf
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\rq1_qber_distribution.pdf


ValueError: x and y must be the same size

### Grafici per RQ3 (Multi-Agent Collaboration)

In [13]:
if not df.empty:
    print("Generazione grafici RQ3...")
    
    p5 = plot_agent_collaboration(df, PLOTS_DIR)
    print(f"  [OK] {p5}")
    
    p6 = plot_handoff_analysis(df, PLOTS_DIR)
    print(f"  [OK] {p6}")
    
    p7 = plot_confidence_analysis(df, PLOTS_DIR)
    print(f"  [OK] {p7}")
else:
    print("Nessun dato per i grafici.")

Generazione grafici RQ3...
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\rq3_agent_collaboration.pdf
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\rq3_handoff_analysis.pdf
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\rq3_confidence_analysis.pdf


### Grafici per RQ5 (Robustness & Reproducibility)

In [14]:
if not df.empty:
    print("Generazione grafici RQ5...")
    
    p8 = plot_robustness_analysis(df, PLOTS_DIR)
    print(f"  [OK] {p8}")
    
    p9 = plot_time_distribution(df, PLOTS_DIR)
    print(f"  [OK] {p9}")
else:
    print("Nessun dato per i grafici.")

Generazione grafici RQ5...
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\rq5_robustness.pdf
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\rq5_time_distribution.pdf


### Grafici cross-cutting

In [15]:
if not df.empty:
    print("Generazione grafici cross-cutting...")
    
    p10 = plot_attack_comparison(df, PLOTS_DIR)
    print(f"  [OK] {p10}")
    
    p11 = plot_correlation_matrix(df, PLOTS_DIR)
    print(f"  [OK] {p11}")
    
    p12 = plot_scenario_summary(df, PLOTS_DIR)
    print(f"  [OK] {p12}")
else:
    print("Nessun dato per i grafici.")

Generazione grafici cross-cutting...
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\rq1_attack_comparison.pdf
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\cross_correlation_matrix.pdf
  [OK] D:\Unsloth\TESI_FINALE\evaluation\plots\cross_scenario_summary.pdf


### Genera tutti i grafici in una volta

In [16]:
if not df.empty:
    paths = generate_all_plots(df, PLOTS_DIR)
    print(f"\n[OK] Generati {len(paths)} grafici:")
    for name, path in paths.items():
        print(f"  [{name}] {path}")
else:
    print("Nessun dato per i grafici.")

ValueError: x and y must be the same size

## 7. Analisi personalizzata

### Filtra per scenario

In [17]:
if not df.empty:
    scenario_name = "intercept_resend"  # Modifica per un altro scenario
    
    filtered = df[df['scenario_name'] == scenario_name]
    valid = filtered[~filtered['crashed']]
    
    print(f"Scenario: {scenario_name}")
    print(f"Trial totali: {len(filtered)}")
    print(f"Trial valide: {len(valid)}")
    print(f"Success rate: {valid['success'].mean():.2%}")
    print(f"QBER mean: {valid['qber'].mean():.4f}")
    print(f"QBER std: {valid['qber'].std():.4f}")
    print(f"Tempo mean: {valid['wall_time_sec'].mean():.2f}s")
    print(f"Tempo std: {valid['wall_time_sec'].std():.2f}s")
    print()
    
    # Distribuzione QBER
    print("Distribuzione QBER:")
    print(valid['qber'].describe())
else:
    print("Nessun dato.")

Scenario: intercept_resend
Trial totali: 50
Trial valide: 50
Success rate: 32.00%
QBER mean: 0.1300
QBER std: 0.0330
Tempo mean: 0.12s
Tempo std: 0.01s

Distribuzione QBER:
count    50.000000
mean      0.129964
std       0.033007
min       0.064981
25%       0.104276
50%       0.134312
75%       0.153852
max       0.200344
Name: qber, dtype: float64


### Confronto scenari

In [18]:
if not df.empty:
    comparison = compute_scenario_comparison(df)
    
    # Crea tabella riassuntiva
    summary_data = {k: v for k, v in comparison.items() if 'status' not in v}
    summary_df = pd.DataFrame(summary_data).T
    summary_df.index.name = 'scenario'
    
    print("CONFRONTO SCENARI:")
    print(summary_df[['success_rate', 'qber_mean', 'qber_std', 'time_mean', 'time_std', 'stealth_mean', 'crash_rate']].round(4).to_string())
else:
    print("Nessun dato.")

CONFRONTO SCENARI:


KeyError: "None of [Index(['success_rate', 'qber_mean', 'qber_std', 'time_mean', 'time_std',\n       'stealth_mean', 'crash_rate'],\n      dtype='object')] are in the [columns]"

### Correlazioni

In [19]:
if not df.empty:
    correlations = compute_correlations(df)
    
    print("CORRELAZIONI CHIAVE:")
    for name, value in correlations.items():
        strength = "strong" if abs(value) > 0.7 else ("moderate" if abs(value) > 0.4 else "weak")
        direction = "positive" if value > 0 else "negative"
        print(f"  {name}: {value:.4f} ({strength} {direction})")
else:
    print("Nessun dato.")

CORRELAZIONI CHIAVE:
  qber_vs_success: -0.5731 (moderate negative)
  time_vs_success: -0.5514 (moderate negative)
  qber_vs_stolen_bits: -0.4482 (moderate negative)
  stealth_vs_success: 0.3593 (weak positive)
  stolen_bits_vs_stealth: 0.3944 (weak positive)


## 8. Esportazione risultati

In [20]:
if not df.empty:
    # Salva in diversi formati
    
    # JSON
    json_path = Path(OUTPUT_DIR) / "results_export.json"
    df.to_json(json_path, indent=2, orient='records', force_ascii=False)
    print(f"[OK] JSON: {json_path}")
    
    # CSV
    csv_path = Path(OUTPUT_DIR) / "results_export.csv"
    df.to_csv(csv_path, index=False)
    print(f"[OK] CSV:  {csv_path}")
    
    # Excel (se openpyxl è installato)
    try:
        excel_path = Path(OUTPUT_DIR) / "results_export.xlsx"
        df.to_excel(excel_path, index=False, engine='openpyxl')
        print(f"[OK] Excel: {excel_path}")
    except ImportError:
        print("[INFO] openpyxl non installato. Installare con: pip install openpyxl")
    
    # Metriche complete
    metrics = compute_metrics(df)
    metrics_path = Path(OUTPUT_DIR) / "metrics_export.json"
    with open(metrics_path, 'w', encoding='utf-8') as f:
        f.write(metrics.to_json(indent=2))
    print(f"[OK] Metrics JSON: {metrics_path}")
else:
    print("Nessun dato da esportare.")

[OK] JSON: D:\Unsloth\TESI_FINALE\evaluation\output\results_export.json
[OK] CSV:  D:\Unsloth\TESI_FINALE\evaluation\output\results_export.csv
[INFO] openpyxl non installato. Installare con: pip install openpyxl


AttributeError: 'dict' object has no attribute 'to_json'

## 9. Riepilogo finale

In [21]:
if not df.empty:
    # Summary finale
    summary = {
        "total_trials": int(len(df)),
        "successful": int((~df['crashed'] & df['success']).sum()),
        "crashed": int(df['crashed'].sum()),
        "success_rate": float((~df['crashed'] & df['success']).mean()),
        "crash_rate": float(df['crashed'].mean()),
        "qber_mean": float(df[~df['crashed']]['qber'].mean()) if len(df[~df['crashed']]) > 0 else 0,
        "qber_std": float(df[~df['crashed']]['qber'].std()) if len(df[~df['crashed']]) > 0 else 0,
        "time_mean": float(df[~df['crashed']]['wall_time_sec'].mean()) if len(df[~df['crashed']]) > 0 else 0,
        "time_std": float(df[~df['crashed']]['wall_time_sec'].std()) if len(df[~df['crashed']]) > 0 else 0,
        "scenarios": df['scenario_name'].nunique(),
        "scenario_list": df['scenario_name'].unique().tolist(),
    }
    
    print("=" * 60)
    print("RIEPILOGO FINALE")
    print("=" * 60)
    for key, value in summary.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.4f}")
        elif isinstance(value, int):
            print(f"  {key}: {value}")
        else:
            print(f"  {key}: {value}")
    print("=" * 60)
else:
    print("Nessun dato disponibile.")

RIEPILOGO FINALE
  total_trials: 400
  successful: 186
  crashed: 0
  success_rate: 0.4650
  crash_rate: 0.0000
  qber_mean: 0.0566
  qber_std: 0.0535
  time_mean: 0.1105
  time_std: 0.0197
  scenarios: 8
  scenario_list: ['intercept_resend', 'intercept_resend_stealth', 'pns', 'blinding', 'trojan_horse', 'qber_tamper', 'mixed', 'clean_baseline']
